# VT_Publication_Analysis

Variable Tone analysis: merged per-mouse CSVs → per-trial metrics, event-locked traces and FIR kernels, saved in one pickle.

**Pipeline:** 1. VT_Preprocessing → **2. VT_Publication_Analysis** → 3. VT_Publication_Figure / 4. VT_SK3_Rescue_Figure

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/VariableTone/RWD/RWD_Processed/` | 24 merged CSVs |

### Outputs (downloaded at the end)
`vt_pub_data.pkl`, for `Photometry/VariableTone/RWD/RWD_Processed/`, containing:
- `vt_df`, `vt_df_combined`: per mouse × tone duration (and pooled) retrieval latency, distance from feeder at tone end, peak and mean dopamine after dispense
- `psth_animals`, `disp_tone_animals`: per-mouse traces around cue, dispense and retrieval
- `trial_psths`, `trial_dist_tone`, `animal_dist_tone`, `hm_time`: per-trial values for the heatmaps and distance panels
- `kern_by_group`, `kern_by_group_split`, `kern_by_animal`, `t_ax`: FIR kernels
- display settings

### How to run
Run all cells top to bottom. The FIR kernel cell is the slow step (several minutes). `SKIP_EXISTING=True` stops if the pickle is already committed; set it to `False` to recompute.

### Methods in brief
- Dispense and retrieval are matched to each tone from the video record, falling back to the photometry TTL when the video missed an event. Early C57 mice have no dispense signal, so their dispense time is the tone end.
- Peak dopamine = maximum z in the 3 s after dispense. Retrieval latency is measured from tone end.
- FIR kernels: −2 to +4 s around each event at 15 Hz, fit by ridge regression (λ = 1). Two models per mouse: tone types pooled, and split by tone duration.

In [ ]:
# @title Setup — imports
# Colab ships pandas / numpy / scipy / statsmodels / tqdm by default; the guard
# below only installs statsmodels if a stripped-down runtime is missing it.
try:
    import statsmodels.api as sm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels.api as sm

import io
import re
import pickle
import requests
import numpy as np
import pandas as pd
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.interpolate import interp1d
from scipy.stats import ttest_ind
from tqdm.notebook import tqdm

print("Imports OK.")


Imports OK.


In [ ]:
# @title Output path & run options
# Inputs are read from the GitHub repo; outputs are written to the Colab runtime
# and downloaded at the end. Commit the outputs to the repo yourself.
import os

GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr" if you point at a personal fork
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_RAW  = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/main"
PKL_URL = f"{GH_RAW}/Photometry/VariableTone/RWD/RWD_Processed/vt_pub_data.pkl"

PKL_OUT = '/content/vt_pub_data.pkl'   # Colab-local; downloaded at the end

# Skip the whole (slow) analysis if the pickle is already committed to the repo.
# True = the finished "run once ever" state. Set False to recompute -- which you
# must do after changing anything upstream, since the FIR fit is the slow step.
SKIP_EXISTING = True
if SKIP_EXISTING and requests.head(PKL_URL, allow_redirects=True).status_code == 200:
    raise SystemExit(
        f"[SKIP] {PKL_URL} already exists on GitHub -- the analysis is already done.\n"
        f"        Set SKIP_EXISTING=False to recompute it from the merged CSVs.")
print(f"Pickle will be written to: {PKL_OUT}")


In [ ]:
# @title Parameters & configuration

# ── The one and only data source: the merged CSVs committed to the repo ───────
# GITHUB_OWNER / GITHUB_REPO are set in the options cell above.
GITHUB_PATH    = "Photometry/VariableTone/RWD/RWD_Processed"
GITHUB_API_URL = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{GITHUB_PATH}"

# ── Groups & display ──────────────────────────────────────────────────────────
# Colors match Pain_Photometry_Analysis.py (the figure notebook applies the final
# publication palette: SHAM #2166AC, SNI #C0392B).
GROUPS = ['SHAM', 'SNI', 'SK3']
GROUP_COLORS = {
    'SHAM': '#4C72B0',
    'SNI':  '#DD8452',
    'SK3':  '#55A868',
}
PRES_FS = {'suptitle': 16, 'title': 13, 'label': 12, 'tick': 11,
           'legend': 10, 'annot': 9, 'sig': 10}

# ── Tone-onset segment window (wide, covers the 20 s tone) ────────────────────
VT_WINDOW_START = -5
VT_WINDOW_END   = 25
VT_NUM_POINTS   = 600
vt_rel_time = np.linspace(VT_WINDOW_START, VT_WINDOW_END, VT_NUM_POINTS)

# ── Event-locked PSTH window (tight, ±5 s around each event) ──────────────────
EVT_WIN_START = -5
EVT_WIN_END   =  5
EVT_POINTS    = 200
event_time    = np.linspace(EVT_WIN_START, EVT_WIN_END, EVT_POINTS)

BONSAI_TRACKING_COLS = ['Speed_cm', 'DistFromFED']

VT_SIGNAL_COLS = {
    'Corrected_Fluorescence_Z_25': ('Z-score',            'green'),
    'Speed_cm':                    ('Speed (cm/s)',        'darkorange'),
    'DistFromFED':                 ('Dist from FED (cm)', 'purple'),
}
VT_EVENT_COLS = {
    'ToneStart_5000':   ('Tone Onset 5s',  '#E05C5C'),
    'ToneStart_10000':  ('Tone Onset 10s', '#E09C3C'),
    'ToneStart_20000':  ('Tone Onset 20s', '#3C7BE0'),
    'Retrieval_Bonsai': ('Retrieval',      'purple'),
}
TONE_COLS      = ['ToneStart_5000', 'ToneStart_10000', 'ToneStart_20000']
TONE_DURATIONS = {'ToneStart_5000': 5, 'ToneStart_10000': 10, 'ToneStart_20000': 20}
TONE_DURS      = [5, 10, 20]

print(f"Source: {GITHUB_API_URL}")


In [ ]:
# @title Helper functions

def parse_group(fname):
    m = re.search(r'(SHAM|SNI|SK3)', fname)
    return m.group(1) if m else 'Unknown'

def get_animal_id(filename):
    m = re.search(r'([FM]\d+)', filename)
    return m.group(1) if m else filename

def get_animal_num(animal_id):
    m = re.search(r'\d+', animal_id)
    return int(m.group()) if m else 999

def extract_segments(event_times, signal, timestamps, rel_time):
    """Extract peri-event segments interpolated onto rel_time."""
    segments = []
    for t in event_times:
        f = interp1d(timestamps, signal, kind='linear',
                     bounds_error=False, fill_value=np.nan)
        segments.append(f(t + rel_time))
    n = len(rel_time)
    return np.array(segments) if segments else np.full((0, n), np.nan)

def mean_sem(segments):
    """Return (mean, sem) ignoring NaNs."""
    if len(segments) == 0:
        n = segments.shape[1] if segments.ndim == 2 else 1
        return np.full(n, np.nan), np.full(n, np.nan)
    m = np.nanmean(segments, axis=0)
    n = np.sum(~np.isnan(segments), axis=0)
    s = np.nanstd(segments, axis=0, ddof=1) / np.sqrt(np.where(n > 1, n, np.nan))
    return m, s

# ── Dispense / retrieval matching (video record, photometry TTL as per-trial fallback)
def get_dispense_times_for_tone(df_raw, tone_col, tone_dur_s, animal_id, tolerance=4.0):
    """Match dispense events to tone onsets, one tone at a time.
    For each tone: nearest Dispense_Bonsai within `tolerance` of the expected time
    (tone onset + duration); if the video missed it, the nearest Dispense_RWD
    (photometry TTL) instead. Tones with neither are skipped. If no tone matches at
    all, dispense is synthesized from tone end ."""
    if tone_col not in df_raw.columns:
        return np.array([])
    tone_times = df_raw.loc[df_raw[tone_col] == 1, 'TimeStampAdjusted'].values
    if len(tone_times) == 0:
        return np.array([])

    # Early C57 animals (num < 5) lack a dispense signal -> synthesize tone_end
    if get_animal_num(animal_id) < 5:
        return tone_times + tone_dur_s

    def _times(col):
        return df_raw.loc[df_raw[col] == 1, 'TimeStampAdjusted'].values if col in df_raw.columns else np.array([])
    disp_b, disp_r = _times('Dispense_Bonsai'), _times('Dispense_RWD')

    def _nearest(times, expected):
        c = times[np.abs(times - expected) <= tolerance]
        return c[np.argmin(np.abs(c - expected))] if len(c) else None

    matched = []
    for t0 in tone_times:
        t = _nearest(disp_b, t0 + tone_dur_s)
        if t is None:
            t = _nearest(disp_r, t0 + tone_dur_s)   # video missed it -> photometry TTL
        if t is not None:
            matched.append(t)
    if matched:
        return np.array(matched)
    return tone_times + tone_dur_s   # last resort: synthesize from tone end

def get_retrieval_times_for_tone(df_raw, tone_col, tone_dur_s, tolerance=4.0, max_wait=15.0):
    """Match retrieval events to tones, one tone at a time.
    For each tone: first Retrieval_Bonsai in [expected - tolerance, expected + max_wait]
    (expected = tone onset + duration); if the video has none there, the first
    Retrieval_RWD (photometry TTL) in the same window. Tones with neither are skipped."""
    if tone_col not in df_raw.columns:
        return np.array([])
    tone_times = df_raw.loc[df_raw[tone_col] == 1, 'TimeStampAdjusted'].values
    if len(tone_times) == 0:
        return np.array([])

    def _times(col):
        return df_raw.loc[df_raw[col] == 1, 'TimeStampAdjusted'].values if col in df_raw.columns else np.array([])
    ret_b, ret_r = _times('Retrieval_Bonsai'), _times('Retrieval_RWD')

    def _first(times, expected):
        c = times[(times >= expected - tolerance) & (times <= expected + max_wait)]
        return c[0] if len(c) else None

    matched = []
    for t0 in tone_times:
        t = _first(ret_b, t0 + tone_dur_s)
        if t is None:
            t = _first(ret_r, t0 + tone_dur_s)      # video missed it -> photometry TTL
        if t is not None:
            matched.append(t)
    return np.array(matched)

print("Helpers defined.")


In [ ]:
# @title Load the merged CSVs from GitHub
# ~388 MB over 24 files, fetched 10 at a time. This is the slow-ish part before
# the FIR fit; it is the price of having no local/Drive state.

def load_file(file_info):
    """Download one merged CSV and tag it with its animal group."""
    r = requests.get(file_info['download_url']); r.raise_for_status()
    return {'filename': file_info['name'], 'group': parse_group(file_info['name']),
            'df': pd.read_csv(io.StringIO(r.text), low_memory=False)}

results = []
response = requests.get(GITHUB_API_URL); response.raise_for_status()
csv_files = [f for f in response.json() if f['name'].endswith('.csv')]
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(load_file, f): f['name'] for f in csv_files}
    with tqdm(total=len(csv_files), desc="Loading CSVs (GitHub)") as pbar:
        for future in as_completed(futures):
            try: results.append(future.result())
            except Exception as e: print(f"  x {futures[future]}: {e}")
            pbar.update(1)

vt_all_data = sorted(results, key=lambda x: x['filename'])
print(f"Loaded {len(vt_all_data)} Variable Tone files from GitHub.")

# ── Data-version guard ────────────────────────────────────────────────────────
# VT_Preprocessing writes DistFromFED_orig (the pre-normalization distance) for every
# mouse with tracking. If it is missing, the processed CSVs were not produced by the
# current VT_Preprocessing and the position and speed metrics would be wrong, so stop.
# Mice without video (F19) have no tracking columns and are exempt.
_tracked = [e for e in vt_all_data if 'Xpos' in e['df'].columns]
_missing = [e['filename'] for e in _tracked
            if 'DistFromFED_orig' not in e['df'].columns]
if _missing:
    raise SystemExit(
        f"[STOP] {len(_missing)} of {len(_tracked)} tracked merged CSVs lack "
        f"'DistFromFED_orig'.\n"
        f"        Run VT_Preprocessing and commit its output to {GITHUB_PATH} first.\n"
        f"        First few: {_missing[:3]}")
print(f"Data-version guard OK — {len(_tracked)} tracked files carry DistFromFED_orig "
      f"({len(vt_all_data) - len(_tracked)} without Bonsai, exempt).")


Loading CSVs (GitHub):   0%|          | 0/24 [00:00<?, ?it/s]

Loaded 24 Variable Tone files from GitHub.
Data-version guard OK — 23 tracked files carry DistFromFED_orig (1 without Bonsai, exempt).


In [ ]:
# @title Build per-animal segment library
# Also computes Speed_cm in place from Xpos/Ypos when the column is missing/empty
# (downstream metrics depend on Speed_cm), and flags has_bonsai per animal.

vt_segments = {}

for entry in vt_all_data:
    fname  = entry['filename']
    group  = entry['group']
    df     = entry['df']
    animal = get_animal_id(fname)

    if 'TimeStampAdjusted' not in df.columns:
        print(f"  Warning: {fname} missing TimeStampAdjusted, skipping.")
        continue

    if ('Xpos' in df.columns and 'Ypos' in df.columns and
            ('Speed_cm' not in df.columns or df['Speed_cm'].isna().all())):
        dx = df['Xpos'].diff()
        dy = df['Ypos'].diff()
        dt = df['TimeStampAdjusted'].diff().replace(0, np.nan)
        df['Speed_cm'] = (np.sqrt(dx**2 + dy**2) / dt).clip(lower=0)
        entry['df'] = df

    timestamps = df['TimeStampAdjusted'].values
    has_bonsai = all(c in df.columns for c in BONSAI_TRACKING_COLS)

    vt_segments[animal] = {'group': group, 'filename': fname,
                           'has_bonsai': has_bonsai, 'events': {}}

    for event_col in VT_EVENT_COLS:
        if event_col not in df.columns:
            continue
        evt_times = df[df[event_col] == 1]['TimeStampAdjusted'].values
        vt_segments[animal]['events'][event_col] = {}
        for sig_col in VT_SIGNAL_COLS:
            if sig_col not in df.columns:
                continue
            segs = extract_segments(evt_times, df[sig_col].values, timestamps, vt_rel_time)
            vt_segments[animal]['events'][event_col][sig_col] = segs

print(f"Built Variable Tone segment library for {len(vt_segments)} animals.")
for animal, data in vt_segments.items():
    bonsai = 'Bonsai' if data['has_bonsai'] else 'no-Bonsai'
    counts = {ev: len(list(data['events'].get(ev, {}).values())[0])
              if data['events'].get(ev) else 0 for ev in VT_EVENT_COLS}
    print(f"  {animal:<10} {data['group']:<6} [{bonsai}]  " +
          "  ".join(f"{ev}: n={n}" for ev, n in counts.items() if n > 0))


Built Variable Tone segment library for 24 animals.
  F18        SHAM   [Bonsai]  ToneStart_5000: n=10  ToneStart_10000: n=7  ToneStart_20000: n=8  Retrieval_Bonsai: n=24
  F19        SHAM   [no-Bonsai]  ToneStart_5000: n=6  ToneStart_10000: n=6  ToneStart_20000: n=5
  F1         SNI    [Bonsai]  ToneStart_5000: n=8  ToneStart_10000: n=6  ToneStart_20000: n=7  Retrieval_Bonsai: n=21
  F26        SNI    [Bonsai]  ToneStart_5000: n=8  ToneStart_10000: n=6  ToneStart_20000: n=7  Retrieval_Bonsai: n=21
  F29        SNI    [Bonsai]  ToneStart_5000: n=7  ToneStart_10000: n=6  ToneStart_20000: n=6  Retrieval_Bonsai: n=20
  F2         SHAM   [Bonsai]  ToneStart_5000: n=8  ToneStart_10000: n=6  ToneStart_20000: n=7  Retrieval_Bonsai: n=21
  F32        SK3    [Bonsai]  ToneStart_5000: n=8  ToneStart_10000: n=6  ToneStart_20000: n=7  Retrieval_Bonsai: n=21
  F33        SK3    [Bonsai]  ToneStart_5000: n=6  ToneStart_10000: n=6  ToneStart_20000: n=6  Retrieval_Bonsai: n=18
  F3         SHAM   [Bon

In [ ]:
# @title Per-trial metrics -> vt_df, vt_df_combined
# Per tone trial: retrieval latency, mean speed during tone, DistFromFED at tone
# end (mean last 2 s), and peak / mean DA in 0-3 s post-dispense.

DA_COL    = 'Corrected_Fluorescence_Z_25'
DA_WIN    = 3.0    # s post-dispense for peak DA
DIST_TAIL = 2.0    # s before tone end for DistFromFED average

def compute_vt_trial_metrics(df, animal_id, tone_col, tone_dur):
    if tone_col not in df.columns:
        return []
    ts         = df['TimeStampAdjusted'].values
    tone_times = df[df[tone_col] == 1]['TimeStampAdjusted'].values
    if len(tone_times) == 0:
        return []
    disp_times = get_dispense_times_for_tone(df, tone_col, tone_dur, animal_id)
    ret_times  = get_retrieval_times_for_tone(df, tone_col, tone_dur,
                                              tolerance=4.0, max_wait=30.0)
    records = []
    for t_tone in tone_times:
        tone_end = t_tone + tone_dur

        rts = ret_times[(ret_times > tone_end) & (ret_times < tone_end + 30)]
        ret_lat = float(rts[0] - tone_end) if len(rts) > 0 else np.nan

        if 'Speed_cm' in df.columns:
            m = (ts >= t_tone) & (ts <= tone_end)
            v = df.loc[m, 'Speed_cm'].dropna().values
            mean_spd = float(np.mean(v)) if len(v) else np.nan
        else:
            mean_spd = np.nan

        if 'DistFromFED' in df.columns:
            m = (ts >= tone_end - DIST_TAIL) & (ts <= tone_end)
            v = df.loc[m, 'DistFromFED'].dropna().values
            dist_end = float(np.mean(v)) if len(v) else np.nan
        else:
            dist_end = np.nan

        disp_match = disp_times[np.abs(disp_times - tone_end) <= 4.0]
        t_disp = disp_match[0] if len(disp_match) else tone_end
        if DA_COL in df.columns:
            m = (ts >= t_disp) & (ts <= t_disp + DA_WIN)
            v = df.loc[m, DA_COL].dropna().values
            peak_da = float(np.nanmax(v))  if len(v) else np.nan
            mean_da = float(np.nanmean(v)) if len(v) else np.nan
        else:
            peak_da = np.nan
            mean_da = np.nan

        records.append({'tone_time': t_tone, 'tone_dur': tone_dur,
                        'ret_latency': ret_lat, 'mean_speed': mean_spd,
                        'dist_at_end': dist_end, 'peak_da': peak_da,
                        'mean_da': mean_da})
    return records

vt_metric_records = []
for entry in vt_all_data:
    animal = get_animal_id(entry['filename'])
    group  = parse_group(entry['filename'])
    df     = entry['df']
    for tone_col in TONE_COLS:
        tone_dur = TONE_DURATIONS[tone_col]
        trials   = compute_vt_trial_metrics(df, animal, tone_col, tone_dur)
        if not trials:
            continue
        def _mean(key):
            v = [t[key] for t in trials if not np.isnan(t[key])]
            return float(np.mean(v)) if v else np.nan
        vt_metric_records.append({
            'animal': animal, 'group': group,
            'tone_col': tone_col, 'tone_dur': tone_dur,
            'ret_latency': _mean('ret_latency'), 'mean_speed': _mean('mean_speed'),
            'dist_at_end': _mean('dist_at_end'), 'peak_da': _mean('peak_da'),
            'mean_da': _mean('mean_da'),
        })

vt_df = pd.DataFrame(vt_metric_records)

VT_METRICS = [
    ('ret_latency', 'Retrieval Latency (s)'),
    ('dist_at_end', 'Dist from FED at Tone End (cm)'),
    ('mean_da',     'Mean Z-score at Dispense'),
    ('peak_da',     'Peak Z-score at Dispense'),
]
vt_df_combined = (vt_df.groupby(['animal', 'group'])[
    [m for m, _ in VT_METRICS]].mean().reset_index())

print(f"VT metric table: {len(vt_df)} rows  "
      f"({vt_df['animal'].nunique()} animals x {vt_df['tone_dur'].nunique()} tone types)")
vt_df.head()


VT metric table: 72 rows  (24 animals x 3 tone types)


,animal,group,tone_col,tone_dur,ret_latency,mean_speed,dist_at_end,peak_da,mean_da
0,F18,SHAM,ToneStart_5000,5,4.332136,8.889020,6.066099,23.364381,7.907911
1,F18,SHAM,ToneStart_10000,10,2.131748,6.912282,2.559985,30.019957,7.747828
2,F18,SHAM,ToneStart_20000,20,2.159059,5.085540,2.736524,29.453648,7.608573
3,F19,SHAM,ToneStart_5000,5,3.965329,NaN,NaN,18.063975,10.658197
4,F19,SHAM,ToneStart_10000,10,2.198413,NaN,NaN,24.835421,12.260496


In [ ]:
# @title Event-locked PSTHs -> psth_animals, disp_tone_animals (SHAM, SNI & SK3)
# psth_animals[event][group] = list of per-animal mean traces (all tones pooled)
# disp_tone_animals[group][dur] = list of per-animal dispense traces, split by tone
# SK3 is included so the rescue figure (Fig 2) has the same PSTHs as the main
# figure; VT_Publication_Figure only indexes PUB_GROUPS=['SHAM','SNI'] and
# ignores the extra key, so the 12-panel figure is unaffected.

_evt_sig = 'Corrected_Fluorescence_Z_25'
_PG_SAVE = list(GROUPS)   # ['SHAM', 'SNI', 'SK3']
_TC_MAP  = {'ToneStart_5000': '5', 'ToneStart_10000': '10', 'ToneStart_20000': '20'}

_EVT_DEFS = [
    ('Cue',       lambda df, tc, ani: df.loc[df[tc] == 1, 'TimeStampAdjusted'].values
                                      if tc in df.columns else np.array([])),
    ('Dispense',  lambda df, tc, ani: get_dispense_times_for_tone(df, tc, TONE_DURATIONS[tc], ani)),
    ('Retrieval', lambda df, tc, ani: get_retrieval_times_for_tone(df, tc, TONE_DURATIONS[tc])),
]

psth_animals      = {ev: {g: [] for g in _PG_SAVE} for ev, _ in _EVT_DEFS}
disp_tone_animals = {g: {'5': [], '10': [], '20': []} for g in _PG_SAVE}

_dfn_save = lambda df, tc, ani: get_dispense_times_for_tone(df, tc, TONE_DURATIONS[tc], ani)

for entry in vt_all_data:
    grp = parse_group(entry['filename'])
    if grp not in _PG_SAVE:
        continue
    df_r = entry['df']; ani = get_animal_id(entry['filename'])
    if _evt_sig not in df_r.columns:
        continue
    ts = df_r['TimeStampAdjusted'].values; sg = df_r[_evt_sig].values
    fi = interp1d(ts, sg, kind='linear', bounds_error=False, fill_value=np.nan)

    for ev, gfn in _EVT_DEFS:
        segs = []
        for tc in TONE_COLS:
            et = gfn(df_r, tc, ani)
            if len(et):
                segs.append(np.array([fi(t + event_time) for t in et]))
        if segs:
            psth_animals[ev][grp].append(np.nanmean(np.vstack(segs), axis=0))

    for tc, dur in _TC_MAP.items():
        if tc not in df_r.columns:
            continue
        et = _dfn_save(df_r, tc, ani)
        if len(et):
            disp_tone_animals[grp][dur].append(
                np.nanmean(np.array([fi(t + event_time) for t in et]), axis=0))

for ev in psth_animals:
    print(f"  {ev:<10} " + "  ".join(f"{g}: {len(psth_animals[ev][g])} animals" for g in _PG_SAVE))


  Cue        SHAM: 8 animals  SNI: 8 animals  SK3: 8 animals
  Dispense   SHAM: 8 animals  SNI: 8 animals  SK3: 8 animals
  Retrieval  SHAM: 8 animals  SNI: 8 animals  SK3: 8 animals


In [ ]:
# @title Per-trial traces for the figure (heatmaps + panel C distances)
# Moved here from VT_Publication_Figure so the figure notebook needs only the
# pickle -- it no longer downloads the 24 merged CSVs (~388 MB) just to rebuild
# these. Computed for ALL THREE groups (the figure uses SHAM/SNI; SK3 is here
# for the rescue figure).
#
# Reuses vt_all_data, already loaded above, so this adds no download.
# NOTE: the heatmap window/resolution is baked in here. Changing _WIN_END or
# _N_PTS means re-running this notebook, not just the figure.

_PRE_HM  = 1.0     # seconds shown before tone onset
_WIN_END = 24.0    # fixed end for all durations -> a 25 s window (-1 to +24)
_N_PTS   = 300     # samples per trial trace after interpolation

_TONE_DEF = {'5':  ('ToneStart_5000',   5),
             '10': ('ToneStart_10000', 10),
             '20': ('ToneStart_20000', 20)}

# trial_psths[dur][group] = [(animal, trace), ...]  -- one entry PER TRIAL
trial_psths      = {d: {g: [] for g in _PG_SAVE} for d in ['5', '10', '20']}
# trial_dist_tone[dur][group] = [float, ...]  -- DistFromFED at exact tone end
trial_dist_tone  = {d: {g: [] for g in _PG_SAVE} for d in ['5', '10', '20']}
# animal_dist_tone[dur][animal] = {'group': g, 'mean': float}
animal_dist_tone = {d: {} for d in ['5', '10', '20']}

_hm_time = np.linspace(-_PRE_HM, _WIN_END, _N_PTS)

for entry in vt_all_data:
    grp = parse_group(entry['filename'])
    if grp not in _PG_SAVE:
        continue
    animal = get_animal_id(entry['filename'])
    df_t   = entry['df']
    if 'Corrected_Fluorescence_Z_25' not in df_t.columns:
        continue

    ts_t = pd.to_numeric(df_t['TimeStampAdjusted'], errors='coerce').values
    z_t  = pd.to_numeric(df_t['Corrected_Fluorescence_Z_25'], errors='coerce').values
    has_dist = 'DistFromFED' in df_t.columns
    dist_t = pd.to_numeric(df_t['DistFromFED'], errors='coerce').values if has_dist else None

    for dur_str, (tc, dur_s) in _TONE_DEF.items():
        if tc not in df_t.columns:
            continue
        tone_onsets = ts_t[pd.to_numeric(df_t[tc], errors='coerce').values == 1]

        # --- per-trial z-score traces for the heatmaps -----------------------
        for t0 in tone_onsets:
            mask  = (ts_t >= t0 - _PRE_HM - 0.5) & (ts_t <= t0 + _WIN_END + 0.5)
            t_rel = ts_t[mask] - t0
            z_win = z_t[mask]
            ok    = np.isfinite(z_win)
            if ok.sum() < 10:
                continue
            trial_psths[dur_str][grp].append(
                (animal, np.interp(_hm_time, t_rel[ok], z_win[ok])))

        # --- distance from FED at the exact tone end (panel C) ---------------
        if has_dist:
            vals = []
            for t0 in tone_onsets:
                v = dist_t[np.argmin(np.abs(ts_t - (t0 + dur_s)))]
                if np.isfinite(v):
                    vals.append(float(v))
            if vals:
                trial_dist_tone[dur_str][grp].extend(vals)
                animal_dist_tone[dur_str][animal] = {'group': grp,
                                                     'mean': float(np.mean(vals))}

print("Per-trial heatmap rows:",
      {d: {g: len(trial_psths[d][g]) for g in _PG_SAVE} for d in ['5', '10', '20']})
print("Panel C animals:",
      {d: len(animal_dist_tone[d]) for d in ['5', '10', '20']})


Per-trial heatmap rows: {'5': {'SHAM': 65, 'SNI': 62, 'SK3': 64}, '10': {'SHAM': 50, 'SNI': 47, 'SK3': 50}, '20': {'SHAM': 55, 'SNI': 53, 'SK3': 53}}
Panel C animals: {'5': 23, '10': 23, '20': 23}


In [ ]:
# @title FIR kernel deconvolution (ridge lambda = 1.0)  --  SLOW STEP
# Two ridge-regularised FIR models per animal:
#   Model A (pooled): Cue, Dispense 5/10/20, Retrieval
#   Model B (split):  Cue 5/10/20, Dispense 5/10/20, Retrieval 5/10/20
# All animals record at 15 Hz; the FIR design (91 lags x 9 events) is rank-
# deficient with only 5-17 events/duration, so ridge lambda=1.0 stabilises it.

PRE_KERN   = 2.0
POST_KERN  = 4.0
KERN_RIDGE = 1.0
KERN_SIG   = 'Corrected_Fluorescence_Z_25'
KERN_EVENTS       = ['Cue', 'Dispense 5', 'Dispense 10', 'Dispense 20', 'Retrieval']
KERN_EVENTS_SPLIT = ['Cue 5', 'Cue 10', 'Cue 20',
                     'Dispense 5', 'Dispense 10', 'Dispense 20',
                     'Retrieval 5', 'Retrieval 10', 'Retrieval 20']

@dataclass
class FIRSpec:
    pre: float
    post: float
    dt: float
    add_intercept: bool = True
    ridge_lambda: float = 0.0

    @property
    def lags(self):
        n_pre  = int(np.round(self.pre  / self.dt))
        n_post = int(np.round(self.post / self.dt))
        return np.arange(-n_pre, n_post + 1, dtype=int)

    @property
    def time_axis(self):
        return self.lags * self.dt

def _build_fir_block(binary_train, lags):
    nT = len(binary_train)
    X  = np.zeros((nT, len(lags)))
    for j, lag in enumerate(lags):
        if lag >= 0:
            X[lag:, j]  = binary_train[:nT - lag]
        else:
            X[:lag, j]  = binary_train[-lag:]
    return X

def _fit_kernels(signal, event_trains, fir_spec):
    """Return dict of {event: kernel_array}, or None if the fit fails."""
    blocks = [_build_fir_block(train, fir_spec.lags) for train in event_trains.values()]
    X = np.column_stack(blocks)
    if fir_spec.add_intercept:
        X = sm.add_constant(X, has_constant='add')
    y = signal.astype(float)
    valid = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    X_v, y_v = X[valid], y[valid]
    # Ridge solution from the normal equations: (X'X + lambda*P) b = X'y, where P is
    # the identity with the intercept left unpenalised. With lambda > 0 the matrix is
    # positive definite, so a direct solve is exact and numerically stable (an SVD-based
    # least-squares fit can fail to converge on some machines for these large designs).
    P = np.eye(X_v.shape[1])
    if fir_spec.add_intercept:
        P[0, 0] = 0.0
    A = X_v.T @ X_v + fir_spec.ridge_lambda * P
    try:
        coef = np.linalg.solve(A, X_v.T @ y_v)
    except np.linalg.LinAlgError:
        return None
    if not np.all(np.isfinite(coef)):
        return None
    beta   = coef[1:] if fir_spec.add_intercept else coef
    n_lags = len(fir_spec.lags)
    return {ev: beta[i * n_lags:(i + 1) * n_lags]
            for i, ev in enumerate(event_trains)}

def _times_to_binary(event_times, time_axis):
    binary = np.zeros(len(time_axis), dtype=float)
    for t in event_times:
        binary[np.argmin(np.abs(time_axis - t))] = 1.0
    return binary

kern_by_group       = {grp: {ev: [] for ev in KERN_EVENTS}       for grp in GROUPS}
kern_by_group_split = {grp: {ev: [] for ev in KERN_EVENTS_SPLIT} for grp in GROUPS}
kern_by_animal      = {}
fir_spec_vt         = None

print("Fitting FIR kernels...")
for entry in tqdm(vt_all_data, desc="FIR fit"):
    animal = get_animal_id(entry['filename'])
    group  = parse_group(entry['filename'])
    df     = entry['df']
    if KERN_SIG not in df.columns or 'TimeStampAdjusted' not in df.columns:
        continue
    ts  = df['TimeStampAdjusted'].values
    sig = df[KERN_SIG].values
    dts = np.diff(ts)
    dt  = float(np.median(dts[dts > 0]))
    if not (np.isfinite(dt) and dt > 0):
        continue
    if fir_spec_vt is None:
        fir_spec_vt = FIRSpec(pre=PRE_KERN, post=POST_KERN, dt=dt, ridge_lambda=KERN_RIDGE)

    cue_times = np.concatenate([
        df.loc[df[tc] == 1, 'TimeStampAdjusted'].values
        for tc in TONE_COLS if tc in df.columns
    ]) if any(tc in df.columns for tc in TONE_COLS) else np.array([])

    disp5  = get_dispense_times_for_tone(df, 'ToneStart_5000',  5,  animal)
    disp10 = get_dispense_times_for_tone(df, 'ToneStart_10000', 10, animal)
    disp20 = get_dispense_times_for_tone(df, 'ToneStart_20000', 20, animal)

    _ret_list = [get_retrieval_times_for_tone(df, tc, TONE_DURATIONS[tc])
                 for tc in TONE_COLS if tc in df.columns]
    ret_times = np.unique(np.concatenate(_ret_list)) if _ret_list else np.array([])

    event_trains = {
        'Cue':         _times_to_binary(cue_times, ts),
        'Dispense 5':  _times_to_binary(disp5,     ts),
        'Dispense 10': _times_to_binary(disp10,    ts),
        'Dispense 20': _times_to_binary(disp20,    ts),
        'Retrieval':   _times_to_binary(ret_times, ts),
    }
    kernels = _fit_kernels(sig, event_trains, fir_spec_vt)
    if kernels is None:
        raise RuntimeError(f"Pooled kernel fit failed for {animal}. Every animal must be "
                           f"fitted, or the group kernels would silently drop it.")
    kern_by_animal[animal] = {'group': group, 'kernels': kernels}
    for ev in KERN_EVENTS:
        if ev in kernels:
            kern_by_group[group][ev].append(kernels[ev])

    ret5  = get_retrieval_times_for_tone(df, 'ToneStart_5000',  5)
    ret10 = get_retrieval_times_for_tone(df, 'ToneStart_10000', 10)
    ret20 = get_retrieval_times_for_tone(df, 'ToneStart_20000', 20)
    cue5  = df.loc[df['ToneStart_5000']  == 1, 'TimeStampAdjusted'].values if 'ToneStart_5000'  in df.columns else np.array([])
    cue10 = df.loc[df['ToneStart_10000'] == 1, 'TimeStampAdjusted'].values if 'ToneStart_10000' in df.columns else np.array([])
    cue20 = df.loc[df['ToneStart_20000'] == 1, 'TimeStampAdjusted'].values if 'ToneStart_20000' in df.columns else np.array([])

    event_trains_split = {
        'Cue 5':        _times_to_binary(cue5,   ts),
        'Cue 10':       _times_to_binary(cue10,  ts),
        'Cue 20':       _times_to_binary(cue20,  ts),
        'Dispense 5':   _times_to_binary(disp5,  ts),
        'Dispense 10':  _times_to_binary(disp10, ts),
        'Dispense 20':  _times_to_binary(disp20, ts),
        'Retrieval 5':  _times_to_binary(ret5,   ts),
        'Retrieval 10': _times_to_binary(ret10,  ts),
        'Retrieval 20': _times_to_binary(ret20,  ts),
    }
    kernels_split = _fit_kernels(sig, event_trains_split, fir_spec_vt)
    if kernels_split is None:
        raise RuntimeError(f"Split kernel fit failed for {animal}. Every animal must be "
                           f"fitted, or the group kernels would silently drop it.")
    for ev in KERN_EVENTS_SPLIT:
        if ev in kernels_split:
            kern_by_group_split[group][ev].append(kernels_split[ev])

t_ax = fir_spec_vt.time_axis
print(f"\nDone. Fit kernels for {len(kern_by_animal)} animals; "
      f"t_ax has {len(t_ax)} lags ({t_ax[0]:.2f} to {t_ax[-1]:.2f} s, dt={fir_spec_vt.dt:.4f}s).")
for g in GROUPS:
    print(f"  {g}: {len(kern_by_group[g]['Cue'])} animals")


In [ ]:
# @title Package outputs, save the pickle & download it

_save = {
    'vt_df':               vt_df,
    'vt_df_combined':      vt_df_combined,
    'event_time':          event_time,
    'psth_animals':        psth_animals,        # {event: {group: [traces]}}
    'disp_tone_animals':   disp_tone_animals,   # {group: {dur: [traces]}}
    'kern_by_group':       kern_by_group,
    'kern_by_group_split': kern_by_group_split,
    'kern_by_animal':      kern_by_animal,
    't_ax':                t_ax,
    'GROUP_COLORS':        GROUP_COLORS,
    'PRES_FS':             PRES_FS,
    # Per-trial figure inputs (so the figure notebook needs no CSVs at all)
    'trial_psths':         trial_psths,        # {dur: {group: [(animal, trace)]}}
    'trial_dist_tone':     trial_dist_tone,    # {dur: {group: [float]}}
    'animal_dist_tone':    animal_dist_tone,   # {dur: {animal: {group, mean}}}
    'hm_time':             _hm_time,           # x-axis for trial_psths traces
}

with open(PKL_OUT, 'wb') as f:
    pickle.dump(_save, f)

print(f"[SAVED] {PKL_OUT}  ({os.path.getsize(PKL_OUT)/1e6:.2f} MB)")
print("Keys:", list(_save.keys()))

# Colab-local output does not survive the runtime -- pull it down, then commit it to
# Photometry/VariableTone/RWD/RWD_Processed/vt_pub_data.pkl yourself. VT_Publication_Figure
# reads it from there.
from google.colab import files
files.download(PKL_OUT)
